In [1]:
import pandas as pd
import numpy as np


In [2]:
df=pd.read_csv('data/UniversalBank.csv')

In [3]:
df.head()

,ID,Age,Experience,Income,ZIP Code,Family,CCAvg,Education,Mortgage,Personal Loan,Securities Account,CD Account,Online,CreditCard
0,1,25,1,49,91107,4,1.6,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1.5,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,1.0,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,2.7,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,1.0,2,0,0,0,0,0,1


In [4]:
df['Experience']=df['Experience'].apply(lambda x: 0 if x<0 else x)

In [5]:
X=df.drop(columns=['ZIP Code','ID','Age','Personal Loan'])
y=df['Personal Loan']

In [6]:
X.head()

,Experience,Income,Family,CCAvg,Education,Mortgage,Securities Account,CD Account,Online,CreditCard
0,1,49,4,1.6,1,0,1,0,0,0
1,19,34,3,1.5,1,0,1,0,0,0
2,15,11,1,1.0,1,0,0,0,0,0
3,9,100,1,2.7,2,0,0,0,0,0
4,8,45,4,1.0,2,0,0,0,0,1


In [7]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: Personal Loan, dtype: int64

In [8]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42)

In [9]:

from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [10]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score, precision_score, recall_score

model = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42)
}

for name,models in model.items():
    models.fit(X_train_resampled, y_train_resampled)
    y_pred = models.predict(X_test_scaled)
    print(f"Model: {name}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
    print(f"F1 Score: {f1_score(y_test, y_pred)}")
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_pred)}")
    print(f"Precision: {precision_score(y_test, y_pred)}")
    print(f"Recall: {recall_score(y_test, y_pred)}")
    print("\n")

Model: Random Forest
Confusion Matrix:
[[892   3]
 [  3 102]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       895
           1       0.97      0.97      0.97       105

    accuracy                           0.99      1000
   macro avg       0.98      0.98      0.98      1000
weighted avg       0.99      0.99      0.99      1000

Accuracy: 0.994
F1 Score: 0.9714285714285714
ROC AUC Score: 0.9840383080606544
Precision: 0.9714285714285714
Recall: 0.9714285714285714


Model: Logistic Regression
Confusion Matrix:
[[803  92]
 [ 10  95]]
Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.90      0.94       895
           1       0.51      0.90      0.65       105

    accuracy                           0.90      1000
   macro avg       0.75      0.90      0.80      1000
weighted avg       0.94      0.90      0.91      1000

Accuracy: 0.898
F1 Score: 0.6506

In [12]:
from sklearn.model_selection import GridSearchCV
rf = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}
grid_search = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1
)
grid_search.fit(X_train_resampled, y_train_resampled)
print("Best parameters found: ", grid_search.best_params_)


Best parameters found:  {'max_depth': 20, 'min_samples_split': 2, 'n_estimators': 50}


In [14]:
best_rf = grid_search.best_estimator_
y_pred_tuned = best_rf.predict(X_test_scaled)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tuned))
print(f"ROC-AUC Score: {roc_auc_score(y_test, best_rf.predict_proba(X_test_scaled)[:, 1]):.4f}")


Confusion Matrix:
[[892   3]
 [  5 100]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       895
           1       0.97      0.95      0.96       105

    accuracy                           0.99      1000
   macro avg       0.98      0.97      0.98      1000
weighted avg       0.99      0.99      0.99      1000

ROC-AUC Score: 0.9989
